# Lesson 4: Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
%pip install langchain_tavily

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [3]:
tool = TavilySearch(max_results=2)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [ ]:
!pip install langgraph-checkpoint-sqlite

In [12]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Create a connection manually to keep it alive across notebook cells
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [13]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [14]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o-mini")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [15]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [16]:
thread = {"configurable": {"thread_id": "1"}}

In [17]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1259, 'total_tokens': 1282, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D0DPSZ13H5DOUhvCqTiXfkaNxoO2S', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bdd46-19e4-7201-94f2-88652d1be29d-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather San Francisco', 'topic': 'news'}, 'id': 'call_pwxWsF52UiPborO8BE1jBHFC', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1259, 'output_tokens': 23, 'total_tokens': 1282, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [18]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 2052, 'total_tokens': 2075, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1920}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D0DPgbQkCaE9cXWO7RWOt4p9WINE5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bdd46-4d82-7673-be74-bb0adab6e1e6-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather Los Angeles', 'topic': 'news'}, 'id': 'call_49nHPe06nNuzueWN4744JMUr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2052, 'output_tokens': 23, 'total_tokens': 2075, 'input_token_details': {'audio': 0, 'cache_read': 1920}, 

In [19]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Currently, Los Angeles is warmer than San Francisco. \n\n- **San Francisco** has temperatures in the upper 50s to low 60s.\n- **Los Angeles** is experiencing temperatures in the low 70s for the valleys and mid-to-upper 60s near the coast.\n\nTherefore, Los Angeles is the warmer city at this time.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 3132, 'total_tokens': 3203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 3072}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D0DPr5Cs1jE7lWX2tgxLi2DRyYVtx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bdd46-7d76-7c00-a51f-3611d87de865-0', tool_calls=[], invalid_tool_calls=[],

In [20]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Could you please provide more details about what you're comparing? Are you asking about the temperature of specific locations, objects, or something else?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 1257, 'total_tokens': 1285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D0DQCfaxbg7pFWZxnJDqKeDKHoj3u', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bdd46-ce72-7523-9bca-46dc3cdb3a7a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1257, 'output_tokens': 28, 'total_tokens': 1285, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_to

## Streaming tokens

In [25]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# In a notebook, manually enter the context to get the actual saver
checkpointer_context = AsyncSqliteSaver.from_conn_string(":memory:")
memory = await checkpointer_context.__aenter__()

# Now you can create the agent as normal
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [26]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_oQBFL9wkj2PvKMhxg5wHaU5f', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| San| Francisco| is| as| follows|:

|-| **|Temperature|**|:| |12|.|8|°C| (|55|°F|)
|-| **|Condition|**|:| Part|ly| cloudy|
|-| **|Wind|**|:| |6|.|9| mph| (|11|.|2| k|ph|)| from| the| North|-N|ort|heast|
|-| **|Humidity|**|:| |72|%
|-| **|Cloud| Cover|**|:| |75|%
|-| **|Visibility|**|:| |16| km| (|9| miles|)
|-| **|UV| Index|**|:| |1|.|7|

|You| can| find| more| details| on| platforms| like| [|Weather| API|](|https|://|www|.weather|api|.com|/)| or| [|Weather| Sh|og|un|](|https|://|we|athers|hog|un|.com|/weather|/|usa|/|ca|/s|an|-fr|anc|isco|/|480|/j|anuary|/|202|6|-|01|-|20|).|